# Information Retrieval 1#
## Assignment 2: Retrieval models [100 points] ##

In [ ]:
import logging
import sys
import os
import matplotlib.pyplot as plt
import numpy as np
import pyndri
import scipy.stats as sp
from scipy.integrate import quad
import time
import itertools
import collections
import io
import logging
import sys
import gensim
from pyndri import compat
from gensim import corpora, models, similarities

index = pyndri.Index('index/')


In this assignment you will get familiar with basic and advanced information retrieval concepts. You will implement different information retrieval ranking models and evaluate their performance.

We provide you with a Indri index. To query the index, you'll use a Python package ([pyndri](https://github.com/cvangysel/pyndri)) that allows easy access to the underlying document statistics.

For evaluation you'll use the [TREC Eval](https://github.com/usnistgov/trec_eval) utility, provided by the National Institute of Standards and Technology of the United States. TREC Eval is the de facto standard way to compute Information Retrieval measures and is frequently referenced in scientific papers.

This is a **groups-of-three assignment**, the deadline is **Wednesday, January 31st**. Code quality, informative comments and convincing analysis of the results will be considered when grading. Submission should be done through blackboard, questions can be asked on the course [Piazza](piazza.com/university_of_amsterdam/spring2018/52041inr6y/home).

### Technicalities (must-read!) ###

The assignment directory is organized as follows:
   * `./assignment.ipynb` (this file): the description of the assignment.
   * `./index/`: the index we prepared for you.
   * `./ap_88_90/`: directory with ground-truth and evaluation sets:
      * `qrel_test`: test query relevance collection (**test set**).
      * `qrel_validation`: validation query relevance collection (**validation set**).
      * `topics_title`: semicolon-separated file with query identifiers and terms.

You will need the following software packages (tested with Python 3.5 inside [Anaconda](https://conda.io/docs/user-guide/install/index.html)):
   * Python 3.5 and Jupyter
   * Indri + Pyndri (Follow the installation instructions [here](https://github.com/nickvosk/pyndri/blob/master/README.md))
   * gensim [link](https://radimrehurek.com/gensim/install.html)
   * TREC Eval [link](https://github.com/usnistgov/trec_eval)

### TREC Eval primer ###
The TREC Eval utility can be downloaded and compiled as follows:

    git clone https://github.com/usnistgov/trec_eval.git
    cd trec_eval
    make

TREC Eval computes evaluation scores given two files: ground-truth information regarding relevant documents, named *query relevance* or *qrel*, and a ranking of documents for a set of queries, referred to as a *run*. The *qrel* will be supplied by us and should not be changed. For every retrieval model (or combinations thereof) you will generate a run of the top-1000 documents for every query. The format of the *run* file is as follows:

    $query_identifier Q0 $document_identifier $rank_of_document_for_query $query_document_similarity $run_identifier
    
where
   * `$query_identifier` is the unique identifier corresponding to a query (usually this follows a sequential numbering).
   * `Q0` is a legacy field that you can ignore.
   * `$document_identifier` corresponds to the unique identifier of a document (e.g., APXXXXXXX where AP denotes the collection and the Xs correspond to a unique numerical identifier).
   * `$rank_of_document_for_query` denotes the rank of the document for the particular query. This field is ignored by TREC Eval and is only maintained for legacy support. The ranks are computed by TREC Eval itself using the `$query_document_similarity` field (see next). However, it remains good practice to correctly compute this field.
   * `$query_document_similarity` is a score indicating the similarity between query and document where a higher score denotes greater similarity.
   * `$run_identifier` is an identifier of the run. This field is for your own convenience and has no purpose beyond bookkeeping.
   
For example, say we have two queries: `Q1` and `Q2` and we rank three documents (`DOC1`, `DOC2`, `DOC3`). For query `Q1`, we find the following similarity scores `score(Q1, DOC1) = 1.0`, `score(Q1, DOC2) = 0.5`, `score(Q1, DOC3) = 0.75`; and for `Q2`: `score(Q2, DOC1) = -0.1`, `score(Q2, DOC2) = 1.25`, `score(Q1, DOC3) = 0.0`. We can generate run using the following snippet:

In [ ]:
def write_run(model_name, data, out_f,
              max_objects_per_query=sys.maxsize,
              skip_sorting=False):
    """
    Write a run to an output file.
    Parameters:
        - model_name: identifier of run.
        - data: dictionary mapping topic_id to object_assesments;
            object_assesments is an iterable (list or tuple) of
            (relevance, object_id) pairs.
            The object_assesments iterable is sorted by decreasing order.
        - out_f: output file stream.
        - max_objects_per_query: cut-off for number of objects per query.
    """
    for subject_id, object_assesments in data.items():
        if not object_assesments:
            logging.warning('Received empty ranking for %s; ignoring.',
                            subject_id)

            continue

        # Probe types, to make sure everything goes alright.
        # assert isinstance(object_assesments[0][0], float) or \
        #     isinstance(object_assesments[0][0], np.float32)
        assert isinstance(object_assesments[0][1], str) or \
            isinstance(object_assesments[0][1], bytes)

        if not skip_sorting:
            object_assesments = sorted(object_assesments, reverse=True)

        if max_objects_per_query < sys.maxsize:
            object_assesments = object_assesments[:max_objects_per_query]

        if isinstance(subject_id, bytes):
            subject_id = subject_id.decode('utf8')

        for rank, (relevance, object_id) in enumerate(object_assesments):
            if isinstance(object_id, bytes):
                object_id = object_id.decode('utf8')

            out_f.write(
                '{subject} Q0 {object} {rank} {relevance} '
                '{model_name}\n'.format(
                    subject=subject_id,
                    object=object_id,
                    rank=rank + 1,
                    relevance=relevance,
                    model_name=model_name))
            
# The following writes the run to standard output.
# In your code, you should write the runs to local
# storage in order to pass them to trec_eval.
# write_run(
#     model_name='example',
#     data={
#         'Q1': ((1.0, 'DOC1'), (0.5, 'DOC2'), (0.75, 'DOC3')),
#         'Q2': ((-0.1, 'DOC1'), (1.25, 'DOC2'), (0.0, 'DOC3')),
#     },
#     out_f=sys.stdout,
#     max_objects_per_query=1000)

In [ ]:
print("There are %d documents in this collection." % (index.maximum_document() - index.document_base()))

In [ ]:
example_document = index.document(index.document_base())
print(example_document[0], example_document[1][:10])

In [ ]:
token2id, id2token, _ = index.get_dictionary()
print(list(id2token.items())[:15])

In [ ]:
print([id2token[word_id] for word_id in example_document[1][:10] if word_id > 0])

In [ ]:
query_tokens = index.tokenize("University of Massachusetts")
print("Query by tokens:", query_tokens)
query_id_tokens = [token2id.get(query_token,0) for query_token in query_tokens]
print("Query by ids with stopwords:", query_id_tokens)
query_id_tokens = [word_id for word_id in query_id_tokens if word_id > 0]
print("Query by ids without stopwords:", query_id_tokens)

### Parsing the query file
You can parse the query file (`ap_88_89/topics_title`) using the following snippet:

In [ ]:
def parse_topics(file_or_files,
                 max_topics=sys.maxsize, delimiter=';'):
    assert max_topics >= 0 or max_topics is None

    topics = collections.OrderedDict()

    if not isinstance(file_or_files, list) and \
            not isinstance(file_or_files, tuple):
        if hasattr(file_or_files, '__iter__'):
            file_or_files = list(file_or_files)
        else:
            file_or_files = [file_or_files]

    for f in file_or_files:
        assert isinstance(f, io.IOBase)

        for line in f:
            assert(isinstance(line, str))

            line = line.strip()

            if not line:
                continue

            topic_id, terms = line.split(delimiter, 1)

            if topic_id in topics and (topics[topic_id] != terms):
                    logging.error('Duplicate topic "%s" (%s vs. %s).',
                                  topic_id,
                                  topics[topic_id],
                                  terms)

            topics[topic_id] = terms

            if max_topics > 0 and len(topics) >= max_topics:
                break

    return topics

# with open('./ap_88_89/topics_title', 'r') as f_topics:
#     print(parse_topics([f_topics]))

### Task 1: Implement and compare lexical IR methods [35 points] ### 

In this task you will implement a number of lexical methods for IR using the **Pyndri** framework. Then you will evaluate these methods on the dataset we have provided using **TREC Eval**.

Use the **Pyndri** framework to get statistics of the documents (term frequency, document frequency, collection frequency; **you are not allowed to use the query functionality of Pyndri**) and implement the following scoring methods in **Python**:

- [TF-IDF](http://nlp.stanford.edu/IR-book/html/htmledition/tf-idf-weighting-1.html) and 
- [BM25](http://nlp.stanford.edu/IR-book/html/htmledition/okapi-bm25-a-non-binary-model-1.html) with k1=1.2 and b=0.75. **[5 points]**
- Language models ([survey](https://drive.google.com/file/d/0B-zklbckv9CHc0c3b245UW90NE0/view))
    - Jelinek-Mercer (explore different values of 𝛌 in the range [0.1, 0.5, 0.9]). **[5 points]**
    - Dirichlet Prior (explore different values of 𝛍 [500, 1000, 1500]). **[5 points]**
    - Absolute discounting (explore different values of 𝛅 in the range [0.1, 0.5, 0.9]). **[5 points]**
    - [Positional Language Models](http://sifaka.cs.uiuc.edu/~ylv2/pub/sigir09-plm.pdf) define a language model for each position of a document, and score a document based on the scores of its PLMs. The PLM is estimated based on propagated counts of words within a document through a proximity-based density function, which both captures proximity heuristics and achieves an effect of “soft” passage retrieval. Implement the PLM, all five kernels, but only the Best position strategy to score documents. Use 𝛔 equal to 50, and Dirichlet smoothing with 𝛍 optimized on the validation set (decide how to optimize this value yourself and motivate your decision in the report). **[10 points]**
    
Implement the above methods and report evaluation measures (on the test set) using the hyper parameter values you optimized on the validation set (also report the values of the hyper parameters). Use TREC Eval to obtain the results and report on `NDCG@10`, Mean Average Precision (`MAP@1000`), `Precision@5` and `Recall@1000`.

For the language models, create plots showing `NDCG@10` with varying values of the parameters. You can do this by chaining small scripts using shell scripting (preferred) or execute trec_eval using Python's `subprocess`.

Compute significance of the results using a [two-tailed paired Student t-test](https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.ttest_rel.html) **[5 points]**. Be wary of false rejection of the null hypothesis caused by the [multiple comparisons problem](https://en.wikipedia.org/wiki/Multiple_comparisons_problem). There are multiple ways to mitigate this problem and it is up to you to choose one.

Analyse the results by identifying specific queries where different methods succeed or fail and discuss possible reasons that cause these differences. This is *very important* in order to understand who the different retrieval functions behave.

**NOTE**: Don’t forget to use log computations in your calculations to avoid underflows. 

**IMPORTANT**: You should structure your code around the helper functions we provide below.

In [ ]:
with open('./ap_88_89/topics_title', 'r') as f_topics:
    queries = parse_topics([f_topics])

index = pyndri.Index('index/')

num_documents = index.maximum_document() - index.document_base()

dictionary = pyndri.extract_dictionary(index)

tokenized_queries = {
    query_id: [dictionary.translate_token(token)
               for token in index.tokenize(query_string)
               if dictionary.has_token(token)]
    for query_id, query_string in queries.items()}

query_term_ids = set(
    query_term_id
    for query_term_ids in tokenized_queries.values()
    for query_term_id in query_term_ids)

print('Gathering statistics about', len(query_term_ids), 'terms.')

# inverted index creation.
start_time = time.time()
document_lengths = {}
unique_terms_per_document = {}
index_int_ext = {}
inverted_index = collections.defaultdict(dict)
collection_frequencies = collections.defaultdict(int)

total_terms = 0

for int_doc_id in range(index.document_base(), index.maximum_document()):
    ext_doc_id, doc_token_ids = index.document(int_doc_id)
    index_int_ext[ext_doc_id] = int_doc_id
    document_bow = collections.Counter(
        token_id for token_id in doc_token_ids
        if token_id > 0)
    document_length = sum(document_bow.values())

    document_lengths[int_doc_id] = document_length
    total_terms += document_length

    unique_terms_per_document[int_doc_id] = len(document_bow)

    for query_term_id in query_term_ids:
        assert query_term_id is not None

        document_term_frequency = document_bow.get(query_term_id, 0)

        if document_term_frequency == 0:
            continue

        collection_frequencies[query_term_id] += document_term_frequency
        inverted_index[query_term_id][int_doc_id] = document_term_frequency

avg_doc_length = total_terms / num_documents
print('Inverted index creation took', time.time() - start_time, 'seconds.')

In [ ]:
def run_retrieval(model_name, score_fn, kernel_func=None):
    """
    Runs a retrieval method for all the queries and writes the TREC-friendly results in a file.
    
    :param model_name: the name of the model (a string)
    :param score_fn: the scoring function (a function - see below for an example) 
    """
    run_out_path = '{}.run'.format(model_name)
    if kernel_func != None:
        run_out_path = '{}'.format(model_name) + str(kernel_func.__name__) + '.run'

    if os.path.exists(run_out_path):
        return

    retrieval_start_time = time.time()

    print('Retrieving using', model_name)
    
    data = {}
    
    if model_name != 'PLM':
    
        # loop over all queries
        for queryid, terms in tokenized_queries.items():
            print(queryid)
            datavalues = list()

            # Loop over all documents
            for int_doc_id in range(index.document_base(), index.maximum_document()):
                document_score = 0

                # loop over all terms in each queries
                for term in terms:
                    # if term occurs in document, add score
                    if int_doc_id in inverted_index[term].keys():
                        document_score += score_fn(int_doc_id, term, len(inverted_index[term]))           
                # only keep relevant scores
                if document_score > 0:
                    datavalues.append((document_score, index.document(int_doc_id)[0]))  
            data[queryid] = datavalues   
            print(queryid)
            
    elif model_name == 'PLM':

         # loop over all queries
        for queryid, terms in tokenized_queries.items():
            print(queryid)

            #create list of internal doc ids for fast looping
            toptfidf_int = []
            for x in toptfidf[queryid]:
                toptfidf_int.append(index_int_ext[x])     
            datavalues = list()

            # Loop over all documents in top of tfidf
            for int_doc_id in toptfidf_int:
                datavalues.append((score_fn(int_doc_id, terms, kernel_func), index.document(int_doc_id)[0]))
            data[queryid] = datavalues
            
    with open(run_out_path, 'w') as f_out:
        write_run(
            model_name=model_name,
            data=data,
            out_f=f_out,
            max_objects_per_query=1000)
                                               
    return


In [ ]:
def tfidf(int_document_id, query_term_id, document_term_freq):
    """
    Scoring function for a document and a query term
    
    :param int_document_id: the document id
    :param query_token_id: the query term id (assuming you have split the query to tokens)
    :param document_term_freq: the document term frequency of the query term 
    """
    
    TF = inverted_index[query_term_id][int_document_id]
    IDF = np.log(num_documents / document_term_freq)
    
    return TF * IDF

# combining the two functions above: 
# run_retrieval('tfidf', tfidf)
# run_retrieval('bm25', bm25)
# run_retrieval('jelinek_mercer', jelinek_mercer)
# run_retrieval('dirichlet_prior', dirichlet_prior)
# run_retrieval('absolute_discounting', absolute_discounting)


In [ ]:
def bm25(int_document_id, query_term_id, document_term_freq):
    """
    Scoring function for a document and a query term
    
    :param int_document_id: the document id
    :param query_token_id: the query term id (assuming you have split the query to tokens)
    :param document_term_freq: the document term frequency of the query term
    
    The queries in the dataset aren't very long, therefore the term with k3 will be left out
    """
    k1 = 1.2
    b = 0.75
    TF = inverted_index[query_term_id][int_document_id]
    IDF = np.log(num_documents / document_term_freq)
    doclen = document_lengths[int_document_id]
    BM25_num = ((k1 + 1) * TF)
    BM25_den = (k1 * ((1 - b) + b * (doclen / avg_doc_length)) + TF) 
    BM25_idf = IDF
    return (BM25_num / BM25_den) * BM25_idf
         

In [ ]:
def jelinek_mercer(int_document_id, query_term_id, document_term_freq):
    """
    Language model scoring function based on Jelinek-Mercer smoothing
    
    :param int_document_id: the document id
    :param query_token_id: the query term id (assuming you have split the query to tokens)
    :param document_term_freq: the document term frequency of the query term
    """
    
    L = 0.9 # [0.1 0.5 0.9]
    TFdoc = inverted_index[query_term_id][int_document_id]
    TFcol = sum(inverted_index[query_term_id].values())
    doclen = document_lengths[int_document_id]
    Clen = total_terms
    
    return L * (TFdoc / doclen) + (1 - L) * (TFcol / total_terms)


In [ ]:
def dirichlet_prior(int_document_id, query_term_id, document_term_frequency):
    """
    Language model scoring function based on Dirichlet prior smoothing
    
    :param int_document_id: the document id
    :param query_token_id: the query term id (assuming you have split the query to tokens)
    :param document_term_freq: the document term frequency of the query term
    """
    mu = 1500 # [500 1000 1500]
    TFdoc = inverted_index[query_term_id][int_document_id]
    TFcol = sum(inverted_index[query_term_id].values())
    doclen = document_lengths[int_document_id]
    Clen = total_terms
    
    return (TFdoc + mu * TFcol / Clen) / (doclen + mu)

In [ ]:
def absolute_discounting(int_document_id, query_term_id, document_term_frequency):
    """
    Language model scoring function based on absolute discounting smoothing
    
    :param int_document_id: the document id
    :param query_token_id: the query term id (assuming you have split the query to tokens)
    :param document_term_freq: the document term frequency of the query term
    """
    
    delta = 0.9 # [0.1 0.5 0.9]
    TFdoc = inverted_index[query_term_id][int_document_id]
    TFcol = sum(inverted_index[query_term_id].values())
    doclen = document_lengths[int_document_id]
    Clen = total_terms
    uniqueterms = unique_terms_per_document[int_document_id]
    
    return (max(TFdoc - delta, 0)) / doclen + (delta * uniqueterms) / doclen * (TFcol / Clen)

In [ ]:
def PLM(int_document_id, query_term_ids, kernel_func):
    """
    Position-based language model. Proximity of query terms in the document lead to higher scores
    
    :param int_document_id: the document id
    :param query_token_id: the query term id (assuming you have split the query to tokens)
    :param document_term_freq: the document term frequency of the query term
    """
    sigma = 50
    SQDi = -1e99
    mu = 500
    ixs = {}
    for query_term_id in query_term_ids:
        ixs[query_term_id] = [j for j, x in enumerate(index.document(int_document_id)[1]) if x == query_term_id]
        
    for i, _ in enumerate(index.document(int_document_id)[1]):
        
        pwDi = []
        pwQs = []
        sumcprime = 0
        score = 0
        # calculate the sum over cprime for the virtual document
        sumcprime = globals()[kernel_func.__name__ + '_prime'](i, len(index.document(int_document_id)[1]), sigma) 
        # for every query term calculate pwDi, all positions it occurs in the doc, cprime and smoothed model
        for query_term_id in query_term_ids:
            pwC = sum(inverted_index[query_term_id].values()) / total_terms
            cprime = 0            
            for ix in ixs[query_term_id]:
                cprime += kernel_func(i, ix, sigma)
                
            pwQs.append(query_term_ids.count(query_term_id))            
            pwDi.append((cprime + mu * pwC) / (sumcprime + mu))
            
        pwQs = [k / sum(pwQs) for k in pwQs]
                
        for k, pwQ in enumerate(pwQs):
            score += pwQ * np.log(pwQ / pwDi[k])
        score = - score
        SQDi = max(SQDi, score)
                    
    return SQDi
                

In [ ]:
kernel_functions = [gaussian, triangle, hamming, circle, passage]


# to save calculation time, only use top 30 docs from tfidf
toptfidf = {}
for queryid in tokenized_queries.keys():
    toptfidf[queryid] = []

with open('tfidf.run') as tdidfdocs:
    for line in tdidfdocs:
        line = line.split()
        if int(line[3]) > 10:
            continue
        toptfidf[line[0]].append(line[2])
        

for func in kernel_functions:
    run_retrieval('PLM', PLM, kernel_func=func)

In [ ]:
# Kernel Functions

def gaussian(i, j, sigma):
    return np.exp((- (i - j) ** 2) / (2 * sigma ** 2))
    
def triangle(i, j, sigma):
    if abs(i - j) <= sigma:
        return 1 - abs(i - j) / sigma
    return 0
                
    
def hamming(i, j, sigma):
    if abs(i - j) <= sigma:  
        return (1 / 2) * (1 + np.cos(abs(i - j) * np.pi / sigma))
    return 0

def circle(i, j, sigma):
    if abs(i - j) <= sigma:       
        return np.sqrt(1 - (abs(i - j) / sigma) ** 2)
    return 0
                  
def passage(i, j, sigma):
    if abs(i - j) <= sigma:  
        return 1
    return 0

def gaussian_prime(i, N, sigma):
    return np.sqrt(2 * np.pi * (sigma ** 2)) * (sp.norm.cdf((N - i) / sigma) - sp.norm.cdf((1 - i) / sigma))
    #return quad(lambda x: np.exp((- (i - x) ** 2) / (2 * sigma ** 2)), 1, N)[0]

def triangle_prime(i, N, sigma):
    return quad(lambda x: 1 - abs(i - x) / sigma if abs(i - x) < sigma else 0, 1, N)[0]

def hamming_prime(i, N, sigma):
    return quad(lambda x: (1 / 2) * (1 + np.cos((abs(i - x) * np.pi) / sigma)) if abs(i - x) < sigma else 0, 1, N)[0]
    
def circle_prime(i, N, sigma):
    return quad(lambda x: np.sqrt(1 - (abs(i - x) / sigma) ** 2) if abs(i - x) < sigma else 0, 1, N)[0]

def passage_prime(i, N, sigma):
    return N

In [ ]:
!trec_eval/trec_eval -m all_trec -q ap_88_89/qrel_test jelinek_mercer_0.1.run | grep -E "^ndcg_cut_10\s"

In [ ]:
def evaluate(modelname, qrel_set, measure, request_variance = False):
    evaluations = !trec_eval/trec_eval -m all_trec -q ap_88_89/{qrel_set} {modelname}
    evaluations = [evaluation.split("\t") for evaluation in evaluations]
    all_values = []
    mean_value = 0
    for line in evaluations:
        if line[0].strip() == measure and line[1] == "all" and request_variance == False:
            mean_value =  float(line[2])
        if line[0].strip() == measure and line[1] != "all" and request_variance == True:
            all_values.append(float(line[2]))
    return mean_value, all_values        


In [ ]:
L = [0.1, 0.5, 0.9]
mu = [500, 1000, 1500]
delta = [0.1, 0.5, 0.9]

parameters = [L, mu, delta]
parameters_name = ['L', 'mu', 'delta']
scoringmethod = ['jelinek_mercer', 'dirichlet_prior', 'absolute_discounting']

jel_mer_scores = [evaluate('jelinek_mercer_' + str(x) + '.run', 'qrel_validation', 'ndcg_cut_10')[0] for x in L]
dir_prior_scores = [evaluate('dirichlet_prior_' + str(x) + '.run', 'qrel_validation', 'ndcg_cut_10')[0] for x in mu]
abs_disc_scores = [evaluate('absolute_discounting_' + str(x) + '.run', 'qrel_validation', 'ndcg_cut_10')[0] for x in delta]
plm_gaussian = evaluate('PLMgaussian.run', 'qrel_validation', 'ndcg_cut_10')[0]
plm_triangle = evaluate('PLMtriangle.run', 'qrel_validation', 'ndcg_cut_10')[0]
plm_hamming = evaluate('PLMhamming.run', 'qrel_validation', 'ndcg_cut_10')[0]
plm_circle = evaluate('PLMcircle.run', 'qrel_validation', 'ndcg_cut_10')[0]
plm_passage = evaluate('PLMpassage.run', 'qrel_validation', 'ndcg_cut_10')[0]


scores = [jel_mer_scores, dir_prior_scores, abs_disc_scores]

f, axes = plt.subplots(ncols=3, nrows=1, figsize=(20, 5))
for i, score in enumerate(scores):
    axes[i].plot(parameters[i], score, 'o')
    axes[i].set_title(scoringmethod[i])
    axes[i].set(xlabel=parameters_name[i], ylabel ='NDCG@10')
plt.show()


### Hyperparameter search

- For Jelinek-Mercer, the highest NDCG@10 is for $\lambda$ = 0.1
- For Dirichlet prior, the highest NDCG@10 is for $\mu = 500$
- For absolute discounting, the hightest NDCG@10 is for $\delta = 0.1$

In [ ]:
L = 0.1
mu = 500
delta = 0.1

measures = ['ndcg_cut_10', 'map_cut_1000', 'P_5', 'recall_1000']

tfidf_score = {x: evaluate('tfidf.run', 'qrel_test', x)[0] for x in measures}
bm25_score = {x: evaluate('bm25.run', 'qrel_test', x)[0] for x in measures}
jel_mer_score = {x: evaluate('jelinek_mercer_' + str(L) + '.run', 'qrel_test', x)[0] for x in measures}
dir_prior_score = {x: evaluate('dirichlet_prior_' + str(mu) + '.run', 'qrel_test', x)[0] for x in measures}
abs_disc_score = {x: evaluate('absolute_discounting_' + str(delta) + '.run', 'qrel_test', x)[0] for x in measures}
plm_gaussian_score = {x: evaluate('PLMgaussian.run', 'qrel_test', x)[0] for x in measures}
plm_triangle_score = {x: evaluate('PLMtriangle.run', 'qrel_test', x)[0] for x in measures}
plm_hamming_score = {x: evaluate('PLMhamming.run', 'qrel_test', x)[0] for x in measures}
plm_circle_score = {x: evaluate('PLMcircle.run', 'qrel_test', x)[0] for x in measures}
plm_passage_score = {x: evaluate('PLMpassage.run', 'qrel_test', x)[0] for x in measures}

print('tfidf', tfidf_score, '\n', 'bm25', bm25_score)
print('jel_mer: ',jel_mer_score, '\n', 'dir_prior: ', dir_prior_score, '\n', 'abs_disc: ', abs_disc_score, '\n gaussian', plm_gaussian_score)

tfidf_query = [{x: evaluate('tfidf.run', 'qrel_test', x, request_variance=True)[1] for x in measures},'tfidf'] 
bm25_query = [{x: evaluate('bm25.run', 'qrel_test', x, request_variance=True)[1] for x in measures},'bm25'] 
jel_mer_query = [{x: evaluate('jelinek_mercer_' + str(L) + '.run', 'qrel_test', x, request_variance=True)[1] for x in measures},'jelinek_mercer'] 
dir_prior_query = [{x: evaluate('dirichlet_prior_' + str(mu) + '.run', 'qrel_test', x, request_variance=True)[1] for x in measures}, 'dirichlet_prior']
abs_disc_query = [{x: evaluate('absolute_discounting_' + str(delta) + '.run', 'qrel_test', x, request_variance=True)[1] for x in measures}, 'absolute_discounting']
plm_gaussian_query = [{x: evaluate('PLMgaussian.run', 'qrel_test', x, request_variance=True)[1] for x in measures},'gaussian']
plm_triangle_query = [{x: evaluate('PLMtriangle.run', 'qrel_test', x, request_variance=True)[1] for x in measures},'triangle']
plm_hamming_query = [{x: evaluate('PLMhamming.run', 'qrel_test', x, request_variance=True)[1] for x in measures},'hamming']
plm_circle_query = [{x: evaluate('PLMcircle.run', 'qrel_test', x, request_variance=True)[1] for x in measures},'circle']
plm_passage_query = [{x: evaluate('PLMpassage.run', 'qrel_test', x, request_variance=True)[1] for x in measures},'passage']

In [ ]:
# alpha with Bonferonni correction
alpha = 0.05 / len(measures)

methods_query = [tfidf_query, bm25_query, jel_mer_query, dir_prior_query, abs_disc_query, plm_gaussian_query, plm_triangle_query,\
                plm_hamming_query, plm_circle_query, plm_passage_query]

for method_1, method_2 in list(itertools.combinations(methods_query, 2)):
    alphas_test = []
    sign_measures = []
    for measure in measures:
        t_stat_test, alpha_test = sp.ttest_rel(method_1[0][measure], method_2[0][measure])
        if alpha_test < alpha:
            alphas_test.append(alpha_test)
            sign_measures.append(measure)
    if len(alphas_test) > 0:
        print(method_1[1], 'and', method_2[1], 'are significantly different for', sign_measures)

### Task 2: Latent Semantic Models (LSMs) [15 points] ###

In this task you will experiment with applying distributional semantics methods ([LSI](http://lsa3.colorado.edu/papers/JASIS.lsi.90.pdf) **[5 points]** and [LDA](https://www.cs.princeton.edu/~blei/papers/BleiNgJordan2003.pdf) **[5 points]**) for retrieval.

You do not need to implement LSI or LDA on your own. Instead, you can use [gensim](http://radimrehurek.com/gensim/index.html). An example on how to integrate Pyndri with Gensim for word2vec can be found [here](https://github.com/cvangysel/pyndri/blob/master/examples/word2vec.py). For the remaining latent vector space models, you will need to implement connector classes (such as `IndriSentences`) by yourself.

In order to use a latent semantic model for retrieval, you need to:
   * build a representation of the query **q**,
   * build a representation of the document **d**,
   * calculate the similarity between **q** and **d** (e.g., cosine similarity, KL-divergence).
     
The exact implementation here depends on the latent semantic model you are using. 
   
Each of these LSMs come with various hyperparameters to tune. Make a choice on the parameters, and explicitly mention the reasons that led you to these decisions. You can use the validation set to optimize hyper parameters you see fit; motivate your decisions. In addition, mention clearly how the query/document representations were constructed for each LSM and explain your choices.

In this experiment, you will first obtain an initial top-1000 ranking for each query using TF-IDF in **Task 1**, and then re-rank the documents using the LSMs. Use TREC Eval to obtain the results and report on `NDCG@10`, Mean Average Precision (`MAP@1000`), `Precision@5` and `Recall@1000`.

Perform significance testing **[5 points]** (similar as in Task 1) in the class of semantic matching methods.

In [ ]:
#Run method for the LSI model
def run_retrieval_lsi(model_name, model_hyper):
    """
    Runs a retrieval method for all the queries and writes the TREC-friendly results in a file.
    
    :param model_name: the name of the model (a string)
    :param score_fn: the scoring function (a function - see below for an example) 
    """
    run_out_path = '{}.run'.format(model_name)

    if os.path.exists(run_out_path):
        return

    retrieval_start_time = time.time()

    print('Retrieving using', model_name)
    
    dictionary_run = pyndri.extract_dictionary(index)
    corpus_run = corpora.MmCorpus('./LSI/corpus.mm')

    tfidf_run = models.TfidfModel.load('./LSI/model.tfidf_model')
    corpus_tfidf_run = corpora.MmCorpus('./LSI/corpus_tfidf_LSI.mm')
    
    load_lsi = './LSI/model_numtopics' + str(model_hyper) + '.lsi'
    lsi_run = models.LsiModel.load(load_lsi)
    
    load_corpus_lsi = './LSI/corpus_lsi' + str(model_hyper) + '.mm'
    corpus_lsi_run = corpora.MmCorpus(load_corpus_lsi)
    
    load_index = './LSI/lsi' + str(model_hyper) + '.index'
    index_lsi = similarities.MatrixSimilarity.load(load_index)
                                     
    data = {}
    
    # loop over all queries
    for queryid, terms in tokenized_queries.items():
        datavalues = list()
        
        bow_querie = dictionary_run.doc2bow(terms)
        tfidf_querie = tfidf_run[bow_querie]
        lsi_querie = lsi_run[tfidf_querie]
        
        sims = index_lsi[lsi_querie]
        sims = sorted(enumerate(sims, 1), key=lambda item: -item[1])
        
        datavalues = [(t[1], index.document(t[0])[0]) for t in sims]
       
        data[queryid] = datavalues
                                
        print(queryid)
    
    with open(run_out_path, 'w') as f_out:
        write_run(
            model_name=model_name,
            data=data,
            out_f=f_out,
            max_objects_per_query=1000)
                                               
    return

In [ ]:
#Run method for the LDA model
def run_retrieval_lda(model_name, model_hyper):
    """
    Runs a retrieval method for all the queries and writes the TREC-friendly results in a file.
    
    :param model_name: the name of the model (a string)
    :param score_fn: the scoring function (a function - see below for an example) 
    """
    run_out_path = '{}.run'.format(model_name)

    if os.path.exists(run_out_path):
        return

    retrieval_start_time = time.time()
    
    dictionary_run = pyndri.extract_dictionary(index)
    corpus_run = corpora.MmCorpus('./LSI/corpus.mm')
    
    load_lda = './LDA/model_numtopics' + str(model_hyper) + '.lda'
    lda_run = models.LdaModel.load(load_lda)
    
    load_corpus_lda = './LDA/corpus_lda' + str(model_hyper) + '.mm'
    corpus_lda_run = corpora.MmCorpus(load_corpus_lda)
    
    load_index = './LDA/lda' + str(model_hyper) + '.index'
    index_lda = similarities.MatrixSimilarity.load(load_index)
    
    data = {}
    
    print('Retrieving using', model_name)
        
    # loop over all queries
    for queryid, terms in tokenized_queries.items():
        datavalues = list()
        
        bow_querie = dictionary_run.doc2bow(terms)
        lda_querie = lda_run[bow_querie]
              
        sims = index_lda[lda_querie]
        sims = sorted(enumerate(sims, 1), key=lambda item: -item[1])
        
        datavalues = [(t[1], index.document(t[0])[0]) for t in sims]
        data[queryid] = datavalues
                                
        print(queryid)
    
    with open(run_out_path, 'w') as f_out:
        write_run(
            model_name=model_name,
            data=data,
            out_f=f_out,
            max_objects_per_query=1000)
                                               
    return


In [ ]:
#Create dictionary and docs from index, dictionary.id2token is for use in models
start_time = time.time()

dictionary = pyndri.extract_dictionary(index)
docs = pyndri.compat.IndriSentences(index, dictionary)

class MyCorpus(object):
    def __iter__(self):
        for doc in docs:
            # assume there's one document per line, tokens separated by whitespace
            yield dictionary.doc2bow(doc)

corpus = MyCorpus()
corpora.MmCorpus.serialize('/home/jeroen/Documents/IR/HW2/LSI/corpus.mm', corpus)

print("run time:", time.time()-start_time)

In [ ]:
#Create TFIDF model and save
start_time = time.time()

corpus = corpora.MmCorpus('/home/jeroen/Documents/IR/HW2/LSI/corpus_LSI.mm')
tfidf = models.TfidfModel(corpus)
tfidf.save('/home/jeroen/Documents/IR/HW2/LSI/model.tfidf_model')

print("run time:", time.time()-start_time)

In [ ]:
#Create TFIDF corpus and save 
start_time = time.time()

corpus = corpora.MmCorpus('/home/jeroen/Documents/IR/HW2/LSI/corpus_LSI.mm')
corpus_tfidf = tfidf[corpus]
corpora.MmCorpus.serialize('/home/jeroen/Documents/IR/HW2/LSI/corpus_tfidf_LSI.mm', corpus_tfidf)
print("run time:", time.time()-start_time)

In [ ]:
#Create different LSI models and save
start_time = time.time()

corpus_tfidf = corpora.MmCorpus('/home/jeroen/Documents/IR/HW2/LSI/corpus_tfidf_LSI.mm')

hyper_topics = [200,300,400,500]

for num in hyper_topics:
    start_time = time.time()
    lsi = models.LsiModel(corpus_tfidf, id2word=dictionary.id2token, num_topics=num)
    save_name = '/home/jeroen/Documents/IR/HW2/LSI/model_numtopics' + str(num) + '.lsi'
    lsi.save(save_name)
    print("run time of ", num, " is:", time.time()-start_time)

In [ ]:
#Create corresponding LSI corpus from model and save
hyper_topics = [200,300,400,500]
tfidf = models.TfidfModel.load('./LSI/model.tfidf_model')

for num in hyper_topics:
    start_time = time.time()
    
    load_name = './LSI/model_numtopics' + str(num) + '.lsi'
    lsi = models.LsiModel.load(load_name)
    corpus_lsi = lsi[corpus_tfidf]
    save_name = './LSI/corpus_lsi' + str(num) + '.mm'    
    corpora.MmCorpus.serialize(save_name, corpus_lsi)
    print("run time of ", num, " is:", time.time()-start_time)

In [ ]:
# Create indexes for comparing similarity from models and save
hyper_topics = [200,300,400, 500]

for num in hyper_topics:
    start_time = time.time()
    load_name = './LSI/corpus_lsi' + str(num) + '.mm'   
    corpus_lsi = corpora.MmCorpus(load_name)
    indexes = similarities.MatrixSimilarity(corpus_lsi, num_features = num)
    save_name = './LSI/lsi' + str(num) + '.index' 
    indexes.save(save_name)
    print("run time:", time.time()-start_time)

In [ ]:
#Create different LDA models, save the models, corpus and index
hyper_topics = [50,100,150]

print('loading..')
dictionary = pyndri.extract_dictionary(index)
corpus = corpora.MmCorpus('./LSI/corpus.mm')
print('done')

for num in hyper_topics:
    start_time = time.time()

    lda = models.LdaModel(corpus, id2word=dictionary.id2token, num_topics=num)
    
    save_name1 = './LDA/model_numtopics' + str(num) + '.lda'
    lda.save(save_name1)
    
    corpus_lda = lda[corpus]
    
    save_name2 = './LDA/corpus_lda' + str(num) + '.mm'
    corpora.MmCorpus.serialize(save_name2, corpus_lda)
    
    corpus_lda = corpora.MmCorpus(save_name2)
    
    index_lda = similarities.MatrixSimilarity(corpus_lda, num_features = num)
    
    save_name3 = './LDA/lda' + str(num) + '.index'
    index_lda.save(save_name3)
    
    print("run time of ",num, "was:", time.time()-start_time)

In [ ]:
# run retrieval mode for all models

hyper_topics_lsi = [200,300,400, 500]

hyper_topics_lda = [50,100,150]

for hyper in hyper_topics_lsi:
    name = 'lsi_'+str(hyper)
    run_retrieval_lsi(name, hyper)

for hyper in hyper_topics_lda:
    name = 'lda_'+str(hyper)
    run_retrieval_lda(name, hyper)


In [ ]:
measures = ['ndcg_cut_10']

lsi_200_vali = {x: evaluate('lsi' + str(200) + '.run', 'qrel_validation', x)[0] for x in measures}
lsi_300_vali = {x: evaluate('lsi' + str(300) + '.run', 'qrel_validation', x)[0] for x in measures}
lsi_400_vali = {x: evaluate('lsi' + str(400) + '.run', 'qrel_validation', x)[0] for x in measures}
lsi_500_vali = {x: evaluate('lsi' + str(500) + '.run', 'qrel_validation', x)[0] for x in measures}

lda_50_vali  = {x: evaluate('lda_' + str(50) + '.run', 'qrel_validation', x)[0] for x in measures}
lda_100_vali = {x: evaluate('lda_' + str(100) + '.run', 'qrel_validation', x)[0] for x in measures}
lda_150_vali = {x: evaluate('lda_' + str(150) + '.run', 'qrel_validation', x)[0] for x in measures}

print("lsi", '\n', lsi_200_vali, '\n' , lsi_300_vali , '\n', lsi_400_vali , '\n',lsi_400_vali , '\n', lsi_500_vali)
print("lda", '\n',lda_50_vali, '\n',lda_100_vali, '\n',lda_150_vali)

#### Hyper Parameter Search

The highest NDCG @ 10 for LSI was reached with 500 topics. <br>
The highest NDCG@10 for the LDA model was reached with 150 topics.

In [ ]:
measures = ['ndcg_cut_10', 'map_cut_1000', 'P_5', 'recall_1000']

lsi_500_score = {x: evaluate('lsi' + str(500) + '.run', 'qrel_test', x)[0] for x in measures}
lda_150_score = {x: evaluate('lda_' + str(150) + '.run', 'qrel_test', x)[0] for x in measures}

lsi_500_query = [{x: evaluate('lsi' + str(500) + '.run', 'qrel_test', x, request_variance=True)[1] for x in measures}, 'lsi_500']
lda_150_query = [{x: evaluate('lda_' + str(150) + '.run', 'qrel_test', x, request_variance=True)[1] for x in measures}, 'lda_150']

In [ ]:
# alpha with Bonferonni correction
alpha = 0.05 / len(measures)

methods_query = [lsi_500_query, lda_150_query]

print("lsi 500:", lsi_500_score)
print("lda 150:", lda_150_score)

for method_1, method_2 in list(itertools.combinations(methods_query, 2)):
    alphas_test = []
    sign_measures = []
    for measure in measures:
        t_stat_test, alpha_test = sp.ttest_rel(method_1[0][measure], method_2[0][measure])
        if alpha_test < alpha:
            alphas_test.append(alpha_test)
            sign_measures.append(measure)
    if len(alphas_test) > 0:
        print(method_1[1], 'and', method_2[1], 'are significantly different for', sign_measures)

### Task 3:  Word embeddings for ranking [20 points] (open-ended) ###

First create word embeddings on the corpus we provided using [word2vec](http://arxiv.org/abs/1411.2738) -- [gensim implementation](https://radimrehurek.com/gensim/models/word2vec.html). You should extract the indexed documents using pyndri and provide them to gensim for training a model (see example [here](https://github.com/nickvosk/pyndri/blob/master/examples/word2vec.py)).
   
This is an open-ended task. It is left up you to decide how you will combine word embeddings to derive query and document representations. Note that since we provide the implementation for training word2vec, you will be graded based on your creativity on combining word embeddings for building query and document representations.

Note: If you want to experiment with pre-trained word embeddings on a different corpus, you can use the word embeddings we provide alongside the assignment (./data/reduced_vectors_google.txt.tar.gz). These are the [google word2vec word embeddings](https://code.google.com/archive/p/word2vec/), reduced to only the words that appear in the document collection we use in this assignment.

In [ ]:
#Training the word2vec model
logging.basicConfig(level=logging.INFO)

if len(sys.argv) <= 1:
    logging.error('Usage: python %s <path-to-indri-index>'.format(sys.argv[0]))

    sys.exit(0)

logging.info('Initializing word2vec.')

word2vec_init = gensim.models.Word2Vec(
    size=300,  # Embedding size
    window=5,  # One-sided window size
    sg=True,  # Skip-gram.
    min_count=1,  # Minimum word frequency.
    sample=1e-3,  # Sub-sample threshold.
    hs=False,  # Hierarchical softmax.
    negative=10,  # Number of negative examples.
    iter=1,  # Number of iterations.
    workers=8,  # Number of workers.
)

index = pyndri.Index('index/')

logging.info('Loading vocabulary.')

dictionary = pyndri.extract_dictionary(index)
sentences = pyndri.compat.IndriSentences(index, dictionary)

logging.info('Constructing word2vec vocabulary.')

# Build vocab.
word2vec_init.build_vocab(sentences, trim_rule=None)

models = [word2vec_init]

for epoch in range(1, 5 + 1):
    logging.info('Epoch %d', epoch)

    model = copy.deepcopy(models[-1])
    model.train(sentences, total_examples=model.corpus_count, epochs=model.iter)

    models.append(model)

logging.info('Trained models: %s', models)

In [ ]:
#Save all the six models to files
for index, model in enumerate(models):
    fname = "word2vec" + str(index+1)
    model.save(fname)

In [ ]:
#Load the last model for use
model = gensim.models.Word2Vec.load('word2vec6')

In [ ]:
import scipy as sp

average_similarity = {}
similarities = []

# to save calculation time, only use top 1000 docs from tfidf
toptfidf = {}
for queryid in tokenized_queries.keys():
    toptfidf[queryid] = []

with open('tfidf.run') as tdidfdocs:
    for line in tdidfdocs:
        line = line.split()
        if int(line[3]) > 1000:
            continue
        toptfidf[line[0]].append(line[2])
        
#create list of internal doc ids for fast looping


# for every query
for query_term_id, terms in tokenized_queries.items():
    print(query_term_id)
    toptfidf_int = []
    for x in toptfidf[query_term_id]:
        toptfidf_int.append(index_int_ext[x])   
    average_similarity[query_term_id] = []

    # loop over all documents
    for i, int_doc_id in enumerate(toptfidf_int):
        term_similarity = []
        
        # keep cosine similarities of top10 highest similarities per query term
        for term in terms:
            similarity = [0] * 10
            term_word2vec = model[id2token[term]]
            document = [id2token[word_id] for word_id in index.document(int_doc_id)[1] if word_id > 0]
            
            # loop over all words in the documents and keep top10 highest similarities
            for word in document:
                similaritytemp = 1 - sp.spatial.distance.cosine(model[word], term_word2vec)
                if similaritytemp > min(similarity):
                    similarity.remove(min(similarity))
                    similarity.append(similaritytemp)
                    
            # add average similarity per query term
            term_similarity.append(np.average(similarity))
            
        # keep average similarity of the query term averages
        average_similarity[query_term_id].append((np.average(term_similarity), index.document(int_doc_id)[0]))
    
    
    
    
with open('embeddings_ranking', 'w') as f_out:
    write_run(
        model_name='embeddings_ranking',
        data=average_similarity,
        out_f=f_out,
        max_objects_per_query=1000)

In [ ]:
measures = ['ndcg_cut_10', 'map_cut_1000', 'P_5', 'recall_1000']

embeddings_ranking_score = {x: evaluate('embeddings_ranking.run', 'qrel_test', x)[0] for x in measures}
print("Score for embedding model:", embeddings_ranking_score)

### Task 4: Learning to rank (LTR) [15 points] (open-ended) ###

In this task you will get an introduction into learning to rank for information retrieval.

You can explore different ways for devising features for the model. Obviously, you can use the retrieval methods you implemented in Task 1, Task 2 and Task 3 as features. Think about other features you can use (e.g. query/document length). Creativity on devising new features and providing motivation for them will be taken into account when grading.

For every query, first create a document candidate set using the top-1000 documents using TF-IDF, and subsequently compute features given a query and a document. Note that the feature values of different retrieval methods are likely to be distributed differently.

You are adviced to start some pointwise learning to rank algorithm e.g. logistic regression, implemented in [scikit-learn](http://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LogisticRegression.html).
Train your LTR model using 10-fold cross validation on the test set. More advanced learning to rank algorithms will be appreciated when grading.

In [ ]:
def getscores(modelname, queryid, ext_doc_id):
    with open(modelname) as model:
        for line in model:
            line = line.split()
            if queryid == line[0] and ext_doc_id == line[2]:
                return line[4]

In [ ]:
from collections import defaultdict, Counter, OrderedDict
from sklearn.linear_model import LogisticRegression

# top 1000 docs by TF-IDF
toptfidf = {}
for queryid in tokenized_queries.keys():
    toptfidf[queryid] = []

with open('tfidf.run') as tdidfdocs:
    for line in tdidfdocs:
        line = line.split()
        if int(line[3]) > 1000:
            continue
        toptfidf[line[0]].append(line[2])
        
print('AP890327-0138' in toptfidf['51'])
        
y = defaultdict(lambda: defaultdict())
file = open('./ap_88_89/qrel_test', 'r')

# create y vectors
for line in file:
    line = line.split()
    y[line[0]][line[2]] = int(line[3])
file.close()

ten_fold = int(len(y.keys()) / 10)

all_queries = list(y.keys())

#hier moet nog LSI aan toegevoegd, jij weet welke toch?
models = ['tfidf.run', 'bm25.run', 'lsi500.run']

LTR = [LogisticRegression()] * 10

for i in range(10):
    test_queries = all_queries[i * ten_fold: (i + 1) * ten_fold]
    train_queries = all_queries[0: i * ten_fold] + all_queries[(i + 1) * ten_fold:]
    x_train = []
    y_train = []
    for query_term_id, terms in tokenized_queries.items():
        print(query_term_id)
        for j, ext_doc_id in enumerate(toptfidf[query_term_id]):
            if ext_doc_id in y[query_term_id]:
                features = [getscores(modelname, query_term_id, ext_doc_id) \
                            if getscores(modelname, query_term_id, ext_doc_id) != None \
                            else 0 for modelname in models]
                x_train.append(features)
                y_train.append(y[query_term_id][ext_doc_id])                
    LTR[i].fit(x_train, y_train)
    #dit ook

    ## WEET NIET WAT ER NOG MIST VERDER


In [ ]:
LTR.save('LTR.model')

## 

### Task 5: Write a report [15 points; instant FAIL if not provided] ###

The report should be a PDF file created using the [sigconf ACM template](https://www.acm.org/publications/proceedings-template) and will determine a significant part of your grade.

   * It should explain what you have implemented, motivate your experiments and detail what you expect to learn from them. **[10 points]**
   * Lastly, provide a convincing analysis of your results and conclude the report accordingly. **[10 points]**
      * Do all methods per\form similarly on all queries? Why?
      * Is there a single retrieval model that outperforms all other retrieval models (i.e., silver bullet)?
      * ...

**Hand in the report and your self-contained implementation source files.** Only send us the files that matter, organized in a well-documented zip/tgz file with clear instructions on how to reproduce your results. That is, we want to be able to regenerate all your results with minimal effort. You can assume that the index and ground-truth information is present in the same file structure as the one we have provided.
